In [1]:
import pandas as pd

df = pd.read_csv("Dataset.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (9551, 21)


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [2]:
df["Cuisines"] = df["Cuisines"].fillna(
    df["Cuisines"].mode()[0]
)

print(df["Cuisines"].isnull().sum())

0


In [3]:
df["Recommendation_Features"] = (
    df["Cuisines"].astype(str)
    + " "
    + df["Price range"].astype(str)
)

df[[
    "Restaurant Name",
    "Recommendation_Features"
]].head()

,Restaurant Name,Recommendation_Features
0,Le Petit Souffle,"French, Japanese, Desserts 3"
1,Izakaya Kikufuji,Japanese 3
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian 4"
3,Ooma,"Japanese, Sushi 4"
4,Sambo Kojin,"Japanese, Korean 4"


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(
    df["Recommendation_Features"]
)

print(tfidf_matrix.shape)

(9551, 148)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

print(similarity_matrix.shape)

(9551, 9551)


In [7]:
def recommend_restaurants(cuisine, price_range):

    user_input = cuisine + " " + str(price_range)

    user_vector = tfidf.transform([user_input])

    similarity_scores = cosine_similarity(
        user_vector,
        tfidf_matrix
    )

    similar_indices = similarity_scores.argsort()[0][-10:][::-1]

    recommendations = df.iloc[
        similar_indices
    ][[
        "Restaurant Name",
        "Cuisines",
        "Price range",
        "Aggregate rating"
    ]]

    return recommendations

In [9]:
recommend_restaurants("North Indian", 3)
recommend_restaurants("Chinese", 2)
recommend_restaurants("Cafe", 2)

,Restaurant Name,Cuisines,Price range,Aggregate rating
9543,Dem Karak�_y,Cafe,2,4.5
8741,R.I.P Cafe & Lounge,Cafe,2,3.5
8724,Cafe Coffee Day,Cafe,1,2.7
8605,Barista,Cafe,2,3.0
8906,Hookie Dookie,Cafe,2,0.0
8451,Paddy's Cafe,Cafe,2,3.7
8432,Cafe Coffee Day,Cafe,1,3.6
9517,Timboo Cafe,Cafe,3,4.2
8384,Barista,Cafe,2,2.9
8390,Cafe Coffee Day,Cafe,1,0.0
